# Module 7: Capstone - Building Your Complete DSP Pipeline

## Welcome to the Summit

Congratulations! You've mastered the fundamental building blocks of Digital Signal Processing. Like a musician who has learned scales, chords, and rhythm, you're now ready to compose your own signal processing symphonies.

This capstone module serves three purposes:
1. **Integration** - Combine all techniques into a unified processing pipeline
2. **Application** - Apply your skills to realistic, complex signals
3. **Exploration** - Discover where to go next in your DSP journey

## The Complete Signal Processing Pipeline

Throughout this course, you've built six essential processing stages. Here's how they work together in a production system:

```
[Raw Signal] → [Windowing] → [FFT] → [Scaling] → [STFT] → [Power Analysis] → [Multi-Channel] → [Results]
     ↓             ↓           ↓         ↓          ↓            ↓               ↓              ↓
  Module 1      Module 2    Module 1  Module 3   Module 4    Module 5       Module 6     Insights!
```

Let's build this complete pipeline as a reusable system.

In [28]:
# Import all necessary libraries
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.fft import rfft, rfftfreq
from ipywidgets import interact, interactive, widgets, Layout, VBox, HBox, Tab
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Set consistent plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100

print("✅ Libraries loaded successfully")
print("📊 Ready to build your complete DSP pipeline")

✅ Libraries loaded successfully
📊 Ready to build your complete DSP pipeline


## Part 1: The Unified DSP Processing Class

First, let's create a comprehensive DSP processor that incorporates everything you've learned. This class will serve as a reference implementation you can adapt for your own projects.

In [29]:
class CompleteDSPProcessor:
    """
    A complete DSP processing pipeline incorporating all course modules.
    
    This class demonstrates professional DSP implementation with:
    - Proper windowing (Module 2)
    - Correct FFT scaling (Module 3)
    - STFT for time-frequency analysis (Module 4)
    - Power spectrum analysis (Module 5)
    - Multi-channel processing (Module 6)
    """
    
    def __init__(self, sample_rate, fft_size=1024, window='hann', overlap=0.5):
        """
        Initialize the DSP processor.
        
        Parameters:
        -----------
        sample_rate : float
            Sampling frequency in Hz
        fft_size : int
            FFT size (must be power of 2)
        window : str
            Window function name ('hann', 'hamming', 'blackman', etc.)
        overlap : float
            Overlap fraction for STFT (0 to 0.95)
        """
        self.fs = sample_rate
        self.fft_size = fft_size
        self.overlap = overlap
        self.hop_size = int(fft_size * (1 - overlap))
        
        # Generate window function
        self.window = signal.get_window(window, fft_size)
        self.window_norm = np.sum(self.window**2)
        
        # Pre-calculate frequency bins
        self.freqs = rfftfreq(fft_size, 1/sample_rate)
        
        # Processing statistics
        self.stats = {
            'frames_processed': 0,
            'total_energy': 0,
            'peak_frequency': None,
            'peak_power': None
        }
    
    def calculate_psd(self, signal_data, scaling='density'):
        """
        Calculate Power Spectral Density with proper scaling (Module 3).
        
        Parameters:
        -----------
        signal_data : array
            Input signal
        scaling : str
            'density' for PSD (V²/Hz) or 'spectrum' for power spectrum (V²)
            
        Returns:
        --------
        freqs : array
            Frequency bins
        psd : array
            Power spectral density or power spectrum
        """
        # Apply window
        windowed = signal_data * self.window
        
        # Compute FFT
        fft_vals = rfft(windowed)
        
        # Calculate power
        power = np.abs(fft_vals)**2
        
        # Apply proper scaling
        if scaling == 'density':
            # Power Spectral Density (V²/Hz)
            scale = 2.0 / (self.fs * self.window_norm)
            power[1:-1] *= 2  # Account for negative frequencies
        else:
            # Power Spectrum (V²)
            scale = 2.0 / self.window_norm
            
        return self.freqs, power * scale
    
    def compute_stft(self, signal_data):
        """
        Compute Short-Time Fourier Transform (Module 4).
        
        Returns:
        --------
        times : array
            Time bins
        freqs : array
            Frequency bins
        stft_matrix : array
            Complex STFT matrix
        """
        f, t, Zxx = signal.stft(signal_data, self.fs, 
                                window=self.window,
                                nperseg=self.fft_size,
                                noverlap=int(self.fft_size * self.overlap))
        return t, f, Zxx
    
    def detect_peaks(self, psd, min_height_db=-60, min_distance_hz=10):
        """
        Detect spectral peaks (Module 5).
        
        Parameters:
        -----------
        psd : array
            Power spectral density
        min_height_db : float
            Minimum peak height in dB
        min_distance_hz : float
            Minimum frequency separation between peaks
            
        Returns:
        --------
        peak_info : list of dict
            Information about each detected peak
        """
        # Convert to dB
        psd_db = 10 * np.log10(psd + 1e-12)
        
        # Find peaks
        min_distance_bins = int(min_distance_hz / (self.fs / self.fft_size))
        peaks, properties = signal.find_peaks(psd_db, 
                                             height=min_height_db,
                                             distance=min_distance_bins,
                                             width=1)
        
        # Extract peak information
        peak_info = []
        for i, peak_idx in enumerate(peaks):
            peak_info.append({
                'frequency': self.freqs[peak_idx],
                'power_db': psd_db[peak_idx],
                'power_linear': psd[peak_idx],
                'bin_index': peak_idx,
                'width_hz': properties['widths'][i] * (self.fs / self.fft_size) if 'widths' in properties else None,
                'snr_db': psd_db[peak_idx] - np.median(psd_db)
            })
        
        return peak_info
    
    def process_dual_channel(self, ch1_data, ch2_data):
        """
        Process dual-channel data (Module 6).
        
        Returns:
        --------
        results : dict
            Cross-spectrum, coherence, and phase difference
        """
        # Window both channels
        ch1_windowed = ch1_data * self.window
        ch2_windowed = ch2_data * self.window
        
        # Compute FFTs
        fft1 = rfft(ch1_windowed)
        fft2 = rfft(ch2_windowed)
        
        # Cross-spectrum
        cross_spectrum = fft1 * np.conj(fft2)
        
        # Auto-spectra
        auto1 = np.abs(fft1)**2
        auto2 = np.abs(fft2)**2
        
        # Coherence
        coherence = np.abs(cross_spectrum)**2 / (auto1 * auto2 + 1e-12)
        
        # Phase difference
        phase_diff = np.angle(cross_spectrum)
        
        return {
            'cross_spectrum': cross_spectrum,
            'coherence': coherence,
            'phase_difference': phase_diff,
            'auto_spectrum_ch1': auto1,
            'auto_spectrum_ch2': auto2
        }
    
    def analyze_complete(self, signal_data, channel2_data=None):
        """
        Complete analysis pipeline.
        """
        results = {}
        
        # Single channel analysis
        freqs, psd = self.calculate_psd(signal_data[:self.fft_size])
        results['psd'] = psd
        results['frequencies'] = freqs
        
        # Peak detection
        results['peaks'] = self.detect_peaks(psd)
        
        # STFT
        times, freqs, stft = self.compute_stft(signal_data)
        results['stft'] = {
            'times': times,
            'frequencies': freqs,
            'magnitude': np.abs(stft)
        }
        
        # Dual channel if available
        if channel2_data is not None:
            results['dual_channel'] = self.process_dual_channel(
                signal_data[:self.fft_size], 
                channel2_data[:self.fft_size]
            )
        
        self.stats['frames_processed'] += 1
        
        return results

# Create an instance
processor = CompleteDSPProcessor(sample_rate=48000, fft_size=1024)
print("✅ Complete DSP Processor initialized")
print(f"   Sample rate: {processor.fs} Hz")
print(f"   FFT size: {processor.fft_size}")
print(f"   Frequency resolution: {processor.fs/processor.fft_size:.2f} Hz")

✅ Complete DSP Processor initialized
   Sample rate: 48000 Hz
   FFT size: 1024
   Frequency resolution: 46.88 Hz


## Part 2: Interactive Pipeline Explorer

Now let's create an interactive tool that lets you see how each processing stage affects the signal. This visualization shows the complete pipeline from raw signal to final analysis.

In [31]:
def create_test_signal(fs, duration, components):
    """
    Create a complex test signal with multiple components.
    """
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    signal_sum = np.zeros_like(t)
    
    for comp in components:
        if comp['type'] == 'sine':
            signal_sum += comp['amplitude'] * np.sin(2 * np.pi * comp['frequency'] * t + comp.get('phase', 0))
        elif comp['type'] == 'chirp':
            signal_sum += comp['amplitude'] * signal.chirp(t, comp['f_start'], duration, comp['f_end'])
        elif comp['type'] == 'transient':
            # Add a transient at specified time
            transient_idx = int(comp['time'] * fs)
            width = int(comp['width'] * fs)
            if transient_idx < len(t):
                window = signal.windows.gaussian(width, std=width/6)
                start = max(0, transient_idx - width//2)
                end = min(len(t), start + width)
                signal_sum[start:end] += comp['amplitude'] * window[:end-start] * np.sin(2 * np.pi * comp['frequency'] * t[start:end])
    
    # Add noise
    signal_sum += np.random.randn(len(t)) * 0.05
    
    return t, signal_sum

def pipeline_explorer():
    """Interactive pipeline exploration tool."""
    
    # Control widgets
    signal_type = widgets.Dropdown(
        options=['Simple', 'Complex', 'ULF/VLF-like', 'Challenge'],
        value='Simple',
        description='Signal Type:'
    )
    
    fft_size = widgets.Dropdown(
        options=[256, 512, 1024, 2048, 4096],
        value=1024,
        description='FFT Size:'
    )
    
    window_type = widgets.Dropdown(
        options=['hann', 'hamming', 'blackman', 'tukey', 'boxcar'],
        value='hann',
        description='Window:'
    )
    
    overlap = widgets.FloatSlider(
        value=0.5,
        min=0,
        max=0.9,
        step=0.1,
        description='Overlap:'
    )
    
    show_stage = widgets.SelectMultiple(
        options=['Time Domain', 'Window Effect', 'FFT', 'PSD', 'STFT', 'Peak Detection'],
        value=('Time Domain', 'FFT', 'STFT'),   # was a list; make it a tuple
        description='Show Stages:',
        rows=6
    )
    
    def update_plot(sig_type, fft_sz, win_type, ovlp, stages):
        # Define signal parameters based on type
        fs = 48000
        duration = 1.0
        
        if sig_type == 'Simple':
            components = [
                {'type': 'sine', 'frequency': 50, 'amplitude': 1.0},
                {'type': 'sine', 'frequency': 120, 'amplitude': 0.5},
            ]
        elif sig_type == 'Complex':
            components = [
                {'type': 'sine', 'frequency': 60, 'amplitude': 1.0},
                {'type': 'sine', 'frequency': 180, 'amplitude': 0.3},
                {'type': 'sine', 'frequency': 300, 'amplitude': 0.2},
                {'type': 'transient', 'time': 0.3, 'frequency': 1000, 'amplitude': 2.0, 'width': 0.01},
            ]
        elif sig_type == 'ULF/VLF-like':
            components = [
                {'type': 'sine', 'frequency': 7.83, 'amplitude': 0.1},  # Schumann resonance
                {'type': 'sine', 'frequency': 60, 'amplitude': 2.0},    # Power line
                {'type': 'sine', 'frequency': 120, 'amplitude': 0.5},   # Power line harmonic
                {'type': 'chirp', 'f_start': 100, 'f_end': 2000, 'amplitude': 0.3},  # Sferic
            ]
        else:  # Challenge
            components = [
                {'type': 'sine', 'frequency': 8.3, 'amplitude': 0.05},
                {'type': 'sine', 'frequency': 60, 'amplitude': 1.5},
                {'type': 'sine', 'frequency': 77, 'amplitude': 0.08},
                {'type': 'transient', 'time': 0.2, 'frequency': 500, 'amplitude': 1.0, 'width': 0.005},
                {'type': 'transient', 'time': 0.7, 'frequency': 1500, 'amplitude': 0.5, 'width': 0.003},
                {'type': 'chirp', 'f_start': 200, 'f_end': 50, 'amplitude': 0.2},
            ]
        
        # Generate signal
        t, signal_data = create_test_signal(fs, duration, components)
        
        # Create processor
        processor = CompleteDSPProcessor(fs, fft_sz, win_type, ovlp)

        # Bug Fix
        import types
        _orig_detect_peaks = processor.detect_peaks
        def _safe_detect_peaks(self, psd, min_height_db=-40, min_distance_hz=10):
            bin_hz = self.fs / self.fft_size
            # ensure SciPy's `distance` (in bins) will be >= 1
            min_distance_hz = max(min_distance_hz, bin_hz)
            return _orig_detect_peaks(psd, min_height_db=min_height_db, min_distance_hz=min_distance_hz)
        processor.detect_peaks = types.MethodType(_safe_detect_peaks, processor)
        
        # Perform analysis
        results = processor.analyze_complete(signal_data)
        
        # Create figure with selected stages
        n_plots = len(stages)
        fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3*n_plots))
        if n_plots == 1:
            axes = [axes]
        
        plot_idx = 0
        
        if 'Time Domain' in stages:
            ax = axes[plot_idx]
            ax.plot(t[:2000], signal_data[:2000], 'b-', linewidth=0.5)
            ax.set_title('Stage 1: Raw Time Domain Signal', fontweight='bold')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Amplitude (V)')
            ax.grid(True, alpha=0.3)
            plot_idx += 1
        
        if 'Window Effect' in stages:
            ax = axes[plot_idx]
            # Show original and windowed
            segment = signal_data[:fft_sz]
            windowed = segment * processor.window
            t_segment = np.arange(fft_sz) / fs
            ax.plot(t_segment, segment, 'b-', alpha=0.5, label='Original')
            ax.plot(t_segment, windowed, 'r-', label='Windowed')
            ax.plot(t_segment, processor.window, 'g--', alpha=0.5, label='Window')
            ax.set_title(f'Stage 2: Window Application ({win_type})', fontweight='bold')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Amplitude')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plot_idx += 1
        
        if 'FFT' in stages:
            ax = axes[plot_idx]
            freqs, psd = processor.calculate_psd(signal_data[:fft_sz], scaling='spectrum')
            ax.semilogy(freqs, psd, 'b-')
            ax.set_title('Stage 3: FFT Magnitude Spectrum', fontweight='bold')
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('Magnitude (V²)')
            ax.set_xlim(0, 2000)
            ax.grid(True, alpha=0.3, which='both')
            plot_idx += 1
        
        if 'PSD' in stages:
            ax = axes[plot_idx]
            freqs, psd = processor.calculate_psd(signal_data[:fft_sz], scaling='density')
            ax.semilogy(freqs, psd, 'b-')
            ax.set_title('Stage 4: Power Spectral Density', fontweight='bold')
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('PSD (V²/Hz)')
            ax.set_xlim(0, 2000)
            ax.grid(True, alpha=0.3, which='both')
            plot_idx += 1
        
        if 'STFT' in stages:
            ax = axes[plot_idx]
            stft_data = results['stft']
            magnitude_db = 20 * np.log10(stft_data['magnitude'] + 1e-12)
            im = ax.pcolormesh(stft_data['times'], stft_data['frequencies'], 
                              magnitude_db, shading='gouraud', cmap='viridis',
                              vmin=np.max(magnitude_db) - 60)
            ax.set_title('Stage 5: STFT Spectrogram', fontweight='bold')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Frequency (Hz)')
            ax.set_ylim(0, 2000)
            plt.colorbar(im, ax=ax, label='Power (dB)')
            plot_idx += 1
        
        if 'Peak Detection' in stages:
            ax = axes[plot_idx]
            freqs, psd = processor.calculate_psd(signal_data[:fft_sz])
            peaks = processor.detect_peaks(
                    psd,
                    min_height_db=-40,
                    min_distance_hz=max(10, processor.fs / processor.fft_size)
            )
            
            # Plot PSD
            psd_db = 10 * np.log10(psd + 1e-12)
            ax.plot(freqs, psd_db, 'b-', alpha=0.5)
            
            # Mark peaks
            for peak in peaks:
                ax.plot(peak['frequency'], peak['power_db'], 'ro', markersize=8)
                ax.annotate(f"{peak['frequency']:.1f} Hz{peak['snr_db']:.1f} dB SNR",
                          xy=(peak['frequency'], peak['power_db']),
                          xytext=(10, 10), textcoords='offset points',
                          fontsize=8, color='red',
                          arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))
            
            ax.set_title(f'Stage 6: Peak Detection ({len(peaks)} peaks found)', fontweight='bold')
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('Power (dB)')
            ax.set_xlim(0, 2000)
            ax.grid(True, alpha=0.3)
            plot_idx += 1
        
        plt.tight_layout()
        plt.show()
        
        # Print analysis summary
        print("📊 Analysis Summary:")
        print("="*50)
        print(f"Signal Type: {sig_type}")
        print(f"FFT Size: {fft_sz} samples")
        print(f"Frequency Resolution: {fs/fft_sz:.2f} Hz")
        print(f"Time Resolution: {fft_sz/fs*1000:.1f} ms")
        if results['peaks']:
            print(f"Detected Peaks:")
            for i, peak in enumerate(results['peaks'][:5]):  # Show top 5 peaks
                print(f"  Peak {i+1}: {peak['frequency']:.1f} Hz, SNR: {peak['snr_db']:.1f} dB")
    
    # Create interactive interface
    ui = widgets.VBox([
        widgets.HBox([signal_type, fft_size]),
        widgets.HBox([window_type, overlap]),
        show_stage
    ])
    
    out = widgets.interactive_output(update_plot, 
                                    {'sig_type': signal_type,
                                     'fft_sz': fft_size,
                                     'win_type': window_type,
                                     'ovlp': overlap,
                                     'stages': show_stage})
    
    display(ui, out)

# Launch the explorer
pipeline_explorer()

Output()

## Part 3: Real-World Processing Scenarios

Let's examine how different parameter choices affect real-world signal processing scenarios. This interactive demo shows the impact of your design decisions.

In [23]:
def scenario_comparison():
    """Compare processing approaches for different scenarios."""
    
    scenario = widgets.Dropdown(
        options=['Lightning Detection', 'Schumann Resonance', 'Power Line Monitoring', 'Broadband Analysis'],
        value='Lightning Detection',
        description='Scenario:',
        style={'description_width': 'initial'}
    )
    
    noise_level = widgets.FloatSlider(
        value=0.1,
        min=0,
        max=1.0,
        step=0.1,
        description='Noise Level:'
    )
    
    averaging = widgets.IntSlider(
        value=1,
        min=1,
        max=100,
        step=1,
        description='Averages:'
    )
    
    def update(scen, noise, avg):
        fs = 48000
        duration = 2.0
        t = np.linspace(0, duration, int(fs * duration))
        
        # Create scenario-specific signal
        if scen == 'Lightning Detection':
            # Fast transient with broadband content
            signal_clean = np.zeros_like(t)
            # Add lightning strikes
            for strike_time in [0.3, 0.8, 1.5]:
                idx = int(strike_time * fs)
                if idx < len(t):
                    # Exponentially decaying impulse
                    remaining = len(t) - idx
                    decay = np.exp(-np.arange(remaining) / (0.001 * fs))
                    signal_clean[idx:] += decay * np.random.randn(remaining) * 2
            
            optimal_fft = 256  # Short window for time resolution
            optimal_overlap = 0.9  # High overlap for smooth tracking
            optimal_window = 'hann'
            
        elif scen == 'Schumann Resonance':
            # Very weak, narrow-band signals
            schumann_freqs = [7.83, 14.3, 20.8, 27.3, 33.8]  # First 5 modes
            signal_clean = np.zeros_like(t)
            for f in schumann_freqs:
                signal_clean += 0.01 * np.sin(2 * np.pi * f * t + np.random.rand() * 2 * np.pi)
            
            optimal_fft = 16384  # Very long window for frequency resolution
            optimal_overlap = 0.5
            optimal_window = 'blackman'  # Minimal leakage
            
        elif scen == 'Power Line Monitoring':
            # Strong 60 Hz and harmonics
            signal_clean = np.zeros_like(t)
            for harmonic in range(1, 6):
                amp = 1.0 / harmonic
                signal_clean += amp * np.sin(2 * np.pi * 60 * harmonic * t)
            
            optimal_fft = 4096  # Balance time and frequency
            optimal_overlap = 0.75
            optimal_window = 'hamming'
            
        else:  # Broadband Analysis
            # Mix of everything
            signal_clean = signal.chirp(t, 10, duration, 5000) * 0.3
            signal_clean += 0.5 * np.sin(2 * np.pi * 100 * t)
            signal_clean += 0.2 * np.sin(2 * np.pi * 1000 * t)
            
            optimal_fft = 2048
            optimal_overlap = 0.5
            optimal_window = 'hann'
        
        # Add noise
        signal_noisy = signal_clean + noise * np.random.randn(len(t))
        
        # Create figure
        fig = plt.figure(figsize=(14, 10))
        gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)
        
        # Process with optimal parameters
        processor_opt = CompleteDSPProcessor(fs, optimal_fft, optimal_window, optimal_overlap)
        
        # Process with suboptimal parameters (opposite of optimal)
        subopt_fft = 256 if optimal_fft > 2048 else 8192
        processor_sub = CompleteDSPProcessor(fs, subopt_fft, 'boxcar', 0.0)
        
        # Time domain comparison
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(t[:4000], signal_noisy[:4000], 'b-', alpha=0.5, linewidth=0.5, label='Noisy Signal')
        ax1.plot(t[:4000], signal_clean[:4000], 'r-', linewidth=1, label='Clean Signal')
        ax1.set_title(f'Scenario: {scen} (Noise Level: {noise:.1f})', fontweight='bold', fontsize=12)
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Amplitude (V)')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Optimal spectrogram
        ax2 = fig.add_subplot(gs[1, 0])
        times_opt, freqs_opt, stft_opt = processor_opt.compute_stft(signal_noisy)
        
        # Average if requested
        if avg > 1:
            # Simulate averaging by reducing noise in magnitude
            mag_opt = np.abs(stft_opt)
            noise_reduction = np.sqrt(avg)
            mag_opt = mag_opt + (mag_opt.mean() * 0.1 / noise_reduction)
        else:
            mag_opt = np.abs(stft_opt)
        
        mag_db_opt = 20 * np.log10(mag_opt + 1e-12)
        im1 = ax2.pcolormesh(times_opt, freqs_opt, mag_db_opt, 
                            shading='gouraud', cmap='viridis',
                            vmin=np.max(mag_db_opt) - 60)
        ax2.set_title(f'Optimal Parameters(FFT: {optimal_fft}, Window: {optimal_window}, Overlap: {optimal_overlap:.0%})', 
                     fontweight='bold', fontsize=10)
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel('Frequency (Hz)')
        ax2.set_ylim(0, min(2000, fs/4))
        plt.colorbar(im1, ax=ax2, label='dB')
        
        # Suboptimal spectrogram
        ax3 = fig.add_subplot(gs[1, 1])
        times_sub, freqs_sub, stft_sub = processor_sub.compute_stft(signal_noisy)
        mag_sub = np.abs(stft_sub)
        mag_db_sub = 20 * np.log10(mag_sub + 1e-12)
        im2 = ax3.pcolormesh(times_sub, freqs_sub, mag_db_sub,
                            shading='gouraud', cmap='viridis',
                            vmin=np.max(mag_db_sub) - 60)
        ax3.set_title(f'Suboptimal Parameters(FFT: {subopt_fft}, Window: boxcar, Overlap: 0%)',
                     fontweight='bold', fontsize=10)
        ax3.set_xlabel('Time (s)')
        ax3.set_ylabel('Frequency (Hz)')
        ax3.set_ylim(0, min(2000, fs/4))
        plt.colorbar(im2, ax=ax3, label='dB')
        
        # Resolution comparison
        ax4 = fig.add_subplot(gs[2, :])
        
        # Calculate averaged PSDs
        n_segments = min(avg, len(signal_noisy) // optimal_fft)
        psd_sum_opt = np.zeros(optimal_fft // 2 + 1)
        psd_sum_sub = np.zeros(subopt_fft // 2 + 1)
        
        for i in range(n_segments):
            start = i * optimal_fft // 2
            if start + optimal_fft <= len(signal_noisy):
                f_opt, p_opt = processor_opt.calculate_psd(signal_noisy[start:start+optimal_fft])
                psd_sum_opt += p_opt
        
        for i in range(min(avg, len(signal_noisy) // subopt_fft)):
            start = i * subopt_fft // 2
            if start + subopt_fft <= len(signal_noisy):
                f_sub, p_sub = processor_sub.calculate_psd(signal_noisy[start:start+subopt_fft])
                psd_sum_sub += p_sub
        
        psd_opt_db = 10 * np.log10(psd_sum_opt / n_segments + 1e-12)
        psd_sub_db = 10 * np.log10(psd_sum_sub / min(avg, len(signal_noisy) // subopt_fft) + 1e-12)
        
        ax4.plot(f_opt, psd_opt_db, 'g-', label='Optimal Parameters', linewidth=1.5)
        ax4.plot(f_sub, psd_sub_db, 'r-', alpha=0.7, label='Suboptimal Parameters', linewidth=1)
        ax4.set_title(f'PSD Comparison (Averaged over {n_segments} segments)', fontweight='bold')
        ax4.set_xlabel('Frequency (Hz)')
        ax4.set_ylabel('Power (dB)')
        ax4.set_xlim(0, min(2000, fs/4))
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print recommendations
        print(f"📋 Recommendations for {scen}:")
        print("="*60)
        print(f"✓ FFT Size: {optimal_fft} samples ({optimal_fft/fs*1000:.1f} ms)")
        print(f"✓ Window: {optimal_window}")
        print(f"✓ Overlap: {optimal_overlap:.0%}")
        print(f"✓ Frequency Resolution: {fs/optimal_fft:.2f} Hz")
        print(f"✓ Time Resolution: {optimal_fft/fs*1000*(1-optimal_overlap):.1f} ms")
        print(f"💡 Key Insight: ")
        
        if scen == 'Lightning Detection':
            print("Use short windows to capture transient events precisely in time.")
        elif scen == 'Schumann Resonance':
            print("Use very long windows and averaging to detect weak narrowband signals.")
        elif scen == 'Power Line Monitoring':
            print("Balance time and frequency resolution to track harmonic changes.")
        else:
            print("Adapt parameters based on the dominant signal characteristics.")
    
    interact(update, scen=scenario, noise=noise_level, avg=averaging)

scenario_comparison()

interactive(children=(Dropdown(description='Scenario:', options=('Lightning Detection', 'Schumann Resonance', …

## Part 4: The Final Challenge - Mystery Signal Analysis

Now it's time to test everything you've learned. Below is a complex signal containing multiple components typical of real-world ULF/VLF recordings. Your task is to identify and characterize all the components using the tools and techniques from this course.

### DO NOT READ generate_mystery_signal() NO CHEATING!!!
but still run its cell to see the output message of next the steps

In [9]:
def generate_mystery_signal():
    """Generate a complex mystery signal for analysis."""
    fs = 48000
    duration = 3.0
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    
    # Initialize signal
    mystery = np.zeros_like(t)
    
    # Component 1: Weak continuous signal (natural resonance)
    mystery += 0.02 * np.sin(2 * np.pi * 8.7 * t + np.pi/4)
    
    # Component 2: Power line interference with drift
    freq_drift = 60 + 0.5 * np.sin(2 * np.pi * 0.1 * t)  # Frequency wobble
    phase = 2 * np.pi * np.cumsum(freq_drift) / fs
    mystery += 0.8 * np.sin(phase)
    
    # Component 3: Power line harmonics
    mystery += 0.3 * np.sin(2 * np.pi * 120 * t)
    mystery += 0.15 * np.sin(2 * np.pi * 180 * t)
    
    # Component 4: Intermittent narrowband signal
    mask = (t > 0.5) & (t < 1.2) | (t > 1.8) & (t < 2.3)
    mystery[mask] += 0.1 * np.sin(2 * np.pi * 73.5 * t[mask])
    
    # Component 5: Lightning-like transients
    for strike_time in [0.3, 1.1, 2.1, 2.7]:
        idx = int(strike_time * fs)
        if idx < len(t):
            # Create realistic sferic
            remaining = min(int(0.02 * fs), len(t) - idx)
            t_strike = np.arange(remaining) / fs
            
            # Dispersed pulse (higher frequencies arrive first)
            for f in [2000, 1000, 500, 200]:
                delay = (2000 - f) / 10000  # Dispersion delay
                idx_delayed = idx + int(delay * fs)
                if idx_delayed + remaining <= len(t):
                    envelope = np.exp(-t_strike * 500) * (1000/f)
                    mystery[idx_delayed:idx_delayed+remaining] += envelope * np.sin(2 * np.pi * f * t_strike)
    
    # Component 6: Slowly varying broadband noise
    noise_envelope = 0.05 * (1 + 0.5 * np.sin(2 * np.pi * 0.3 * t))
    mystery += noise_envelope * np.random.randn(len(t))
    
    # Component 7: Mystery chirp (whale-like)
    chirp_mask = (t > 1.4) & (t < 1.7)
    t_chirp = t[chirp_mask]
    mystery[chirp_mask] += 0.15 * signal.chirp(t_chirp - 1.4, 150, 0.3, 50, method='quadratic')
    
    # Add some realistic background noise
    mystery += 0.01 * np.random.randn(len(t))
    
    return t, mystery, fs

# Generate the mystery signal
t_mystery, mystery_signal, fs_mystery = generate_mystery_signal()

print("🔍 FINAL CHALLENGE: Mystery Signal Analysis")
print("="*60)
print("A complex signal has been recorded. Your mission:")
print("1. Identify all signal components")
print("2. Characterize their properties (frequency, timing, strength)")
print("3. Determine which are natural vs. man-made")
print("4. Suggest the optimal processing parameters for each")
print("Use the interactive analysis tool below to investigate...")

🔍 FINAL CHALLENGE: Mystery Signal Analysis
A complex signal has been recorded. Your mission:
1. Identify all signal components
2. Characterize their properties (frequency, timing, strength)
3. Determine which are natural vs. man-made
4. Suggest the optimal processing parameters for each
Use the interactive analysis tool below to investigate...


In [10]:
def mystery_signal_analyzer():
    """Interactive tool for analyzing the mystery signal."""
    
    # Analysis controls
    view_type = widgets.Dropdown(
        options=['Overview', 'Time Zoom', 'Frequency Analysis', 'Spectrogram', 'Peak Detection', 'Component Isolation'],
        value='Overview',
        description='Analysis View:',
        style={'description_width': 'initial'}
    )
    
    time_range = widgets.FloatRangeSlider(
        value=[0, 3],
        min=0,
        max=3,
        step=0.1,
        description='Time Range (s):',
        style={'description_width': 'initial'}
    )
    
    fft_size = widgets.Dropdown(
        options=[512, 1024, 2048, 4096, 8192, 16384],
        value=2048,
        description='FFT Size:'
    )
    
    window_func = widgets.Dropdown(
        options=['hann', 'hamming', 'blackman', 'tukey', 'flattop'],
        value='hann',
        description='Window:'
    )
    
    freq_range = widgets.FloatRangeSlider(
        value=[0, 500],
        min=0,
        max=5000,
        step=10,
        description='Freq Range (Hz):',
        style={'description_width': 'initial'}
    )
    
    def analyze(view, t_range, fft_sz, win, f_range):
        # Extract time range
        t_mask = (t_mystery >= t_range[0]) & (t_mystery <= t_range[1])
        t_sel = t_mystery[t_mask]
        sig_sel = mystery_signal[t_mask]
        
        if view == 'Overview':
            fig, axes = plt.subplots(3, 1, figsize=(12, 9))
            
            # Full time series
            axes[0].plot(t_mystery, mystery_signal, 'b-', linewidth=0.5)
            axes[0].set_title('Complete Mystery Signal', fontweight='bold')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude (V)')
            axes[0].grid(True, alpha=0.3)
            axes[0].axhspan(-0.1, 0.1, alpha=0.2, color='green', label='Weak signal zone')
            
            # Spectrogram
            f, t, Sxx = signal.spectrogram(mystery_signal, fs_mystery, 
                                          nperseg=1024, noverlap=960)
            axes[1].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), 
                             shading='gouraud', cmap='viridis')
            axes[1].set_title('Time-Frequency Overview', fontweight='bold')
            axes[1].set_ylabel('Frequency (Hz)')
            axes[1].set_ylim(0, 500)
            
            # Average spectrum
            freqs, psd = signal.welch(mystery_signal, fs_mystery, nperseg=4096)
            axes[2].semilogy(freqs, psd, 'b-')
            axes[2].set_title('Average Power Spectral Density', fontweight='bold')
            axes[2].set_xlabel('Frequency (Hz)')
            axes[2].set_ylabel('PSD (V²/Hz)')
            axes[2].set_xlim(0, 500)
            axes[2].grid(True, alpha=0.3, which='both')
            
        elif view == 'Time Zoom':
            fig, axes = plt.subplots(2, 1, figsize=(12, 6))
            
            # Zoomed signal
            axes[0].plot(t_sel, sig_sel, 'b-', linewidth=0.7)
            axes[0].set_title(f'Zoomed Time Domain ({t_range[0]:.1f}s - {t_range[1]:.1f}s)', fontweight='bold')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude (V)')
            axes[0].grid(True, alpha=0.3)
            
            # Amplitude statistics
            axes[1].hist(sig_sel, bins=50, density=True, alpha=0.7, color='blue')
            axes[1].axvline(sig_sel.mean(), color='red', linestyle='--', label=f'Mean: {sig_sel.mean():.3f}')
            axes[1].axvline(sig_sel.std(), color='orange', linestyle='--', label=f'Std: {sig_sel.std():.3f}')
            axes[1].set_title('Amplitude Distribution', fontweight='bold')
            axes[1].set_xlabel('Amplitude (V)')
            axes[1].set_ylabel('Probability Density')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            
        elif view == 'Frequency Analysis':
            processor = CompleteDSPProcessor(fs_mystery, fft_sz, win)
            
            # Calculate PSD for selected region
            if len(sig_sel) >= fft_sz:
                freqs, psd = processor.calculate_psd(sig_sel[:fft_sz])
                peaks = processor.detect_peaks(psd, min_height_db=-50)
                
                fig, axes = plt.subplots(2, 1, figsize=(12, 8))
                
                # Linear scale
                axes[0].plot(freqs, psd, 'b-')
                axes[0].set_title(f'Power Spectral Density (Linear Scale)', fontweight='bold')
                axes[0].set_xlabel('Frequency (Hz)')
                axes[0].set_ylabel('PSD (V²/Hz)')
                axes[0].set_xlim(f_range)
                axes[0].grid(True, alpha=0.3)
                
                # Log scale with peaks
                psd_db = 10 * np.log10(psd + 1e-12)
                axes[1].plot(freqs, psd_db, 'b-', alpha=0.7)
                
                # Mark peaks
                for peak in peaks:
                    if f_range[0] <= peak['frequency'] <= f_range[1]:
                        axes[1].plot(peak['frequency'], peak['power_db'], 'ro', markersize=8)
                        axes[1].text(peak['frequency'], peak['power_db'] + 2, 
                                   f"{peak['frequency']:.1f} Hz
                                   fontsize=8, ha='center')
                
                axes[1].set_title(f'Power Spectral Density (Log Scale) - {len(peaks)} peaks detected', fontweight='bold')
                axes[1].set_xlabel('Frequency (Hz)')
                axes[1].set_ylabel('PSD (dB)')
                axes[1].set_xlim(f_range)
                axes[1].grid(True, alpha=0.3)
                
        elif view == 'Spectrogram':
            fig, ax = plt.subplots(1, 1, figsize=(12, 6))
            
            f, t, Sxx = signal.spectrogram(sig_sel, fs_mystery,
                                          window=win,
                                          nperseg=fft_sz,
                                          noverlap=int(fft_sz * 0.9))
            
            # Plot in dB
            Sxx_db = 10 * np.log10(Sxx + 1e-12)
            im = ax.pcolormesh(t + t_range[0], f, Sxx_db,
                              shading='gouraud', cmap='viridis',
                              vmin=np.max(Sxx_db) - 60)
            
            ax.set_title(f'Detailed Spectrogram (FFT: {fft_sz}, Window: {win})', fontweight='bold')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Frequency (Hz)')
            ax.set_ylim(f_range)
            plt.colorbar(im, ax=ax, label='Power (dB)')
            
        elif view == 'Peak Detection':
            # Multi-resolution peak detection
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            axes = axes.flatten()
            
            resolutions = [1024, 4096, 16384, 65536]
            
            for i, res in enumerate(resolutions):
                if res <= len(mystery_signal):
                    f, psd = signal.welch(mystery_signal, fs_mystery, nperseg=min(res, len(mystery_signal)//4))
                    
                    # Find peaks
                    psd_db = 10 * np.log10(psd + 1e-12)
                    peaks, _ = signal.find_peaks(psd_db, height=-60, distance=5)
                    
                    axes[i].semilogy(f, psd, 'b-', alpha=0.5)
                    axes[i].semilogy(f[peaks], psd[peaks], 'ro', markersize=6)
                    
                    # Annotate main peaks
                    for peak_idx in peaks[:5]:  # Top 5 peaks
                        axes[i].annotate(f'{f[peak_idx]:.1f}',
                                       xy=(f[peak_idx], psd[peak_idx]),
                                       fontsize=7)
                    
                    axes[i].set_title(f'Resolution: {fs_mystery/res:.2f} Hz', fontsize=10)
                    axes[i].set_xlabel('Frequency (Hz)')
                    axes[i].set_ylabel('PSD (V²/Hz)')
                    axes[i].set_xlim(0, 500)
                    axes[i].grid(True, alpha=0.3)
            
        elif view == 'Component Isolation':
            # Apply different filters to isolate components
            fig, axes = plt.subplots(4, 1, figsize=(12, 10))
            
            # Low frequency (< 20 Hz)
            sos_low = signal.butter(4, 20, 'low', fs=fs_mystery, output='sos')
            sig_low = signal.sosfilt(sos_low, sig_sel)
            axes[0].plot(t_sel, sig_low, 'b-', linewidth=0.7)
            axes[0].set_title('Low Frequency Components (< 20 Hz) - Natural Resonances', fontweight='bold')
            axes[0].set_ylabel('Amplitude (V)')
            axes[0].grid(True, alpha=0.3)
            
            # Power line (50-70 Hz)
            sos_power = signal.butter(4, [50, 70], 'band', fs=fs_mystery, output='sos')
            sig_power = signal.sosfilt(sos_power, sig_sel)
            axes[1].plot(t_sel, sig_power, 'r-', linewidth=0.7)
            axes[1].set_title('Power Line Band (50-70 Hz)', fontweight='bold')
            axes[1].set_ylabel('Amplitude (V)')
            axes[1].grid(True, alpha=0.3)
            
            # Mid frequency (70-200 Hz)
            sos_mid = signal.butter(4, [70, 200], 'band', fs=fs_mystery, output='sos')
            sig_mid = signal.sosfilt(sos_mid, sig_sel)
            axes[2].plot(t_sel, sig_mid, 'g-', linewidth=0.7)
            axes[2].set_title('Mid Frequency (70-200 Hz) - Harmonics & Unknowns', fontweight='bold')
            axes[2].set_ylabel('Amplitude (V)')
            axes[2].grid(True, alpha=0.3)
            
            # High frequency (> 500 Hz)
            sos_high = signal.butter(4, 500, 'high', fs=fs_mystery, output='sos')
            sig_high = signal.sosfilt(sos_high, sig_sel)
            axes[3].plot(t_sel, sig_high, 'm-', linewidth=0.7)
            axes[3].set_title('High Frequency (> 500 Hz) - Transients & Sferics', fontweight='bold')
            axes[3].set_xlabel('Time (s)')
            axes[3].set_ylabel('Amplitude (V)')
            axes[3].grid(True, alpha=0.3)
            
        plt.tight_layout()
        plt.show()
    
    # Create interface
    ui = widgets.VBox([
        view_type,
        widgets.HBox([fft_size, window_func]),
        time_range,
        freq_range
    ])
    
    out = widgets.interactive_output(analyze,
                                    {'view': view_type,
                                     't_range': time_range,
                                     'fft_sz': fft_size,
                                     'win': window_func,
                                     'f_range': freq_range})
    
    display(ui, out)

mystery_signal_analyzer()

Output()

## Your Analysis Report

Based on your investigation, fill in your findings below. This is what a real DSP analysis report might look like:

In [15]:
# Run this cell to see the solution and compare with your analysis

def reveal_solution():
    print("📋 MYSTERY SIGNAL ANALYSIS SOLUTION")
    print("="*70)
    print("🎯 IDENTIFIED COMPONENTS:")
    
    components = [
        {
            "name": "Natural Resonance",
            "frequency": "8.7 Hz",
            "amplitude": "20 mV",
            "type": "Continuous",
            "nature": "Natural (possibly Schumann-like)",
            "detection": "Requires long FFT (16384+) and averaging"
        },
        {
            "name": "Power Line Fundamental",
            "frequency": "60 Hz (with ±0.5 Hz drift)",
            "amplitude": "800 mV",
            "type": "Continuous with frequency modulation",
            "nature": "Man-made interference",
            "detection": "Easily visible in any FFT size"
        },
        {
            "name": "Power Line Harmonics",
            "frequency": "120 Hz, 180 Hz",
            "amplitude": "300 mV, 150 mV",
            "type": "Continuous",
            "nature": "Man-made (harmonic distortion)",
            "detection": "Clear in frequency domain"
        },
        {
            "name": "Unknown Narrowband Signal",
            "frequency": "73.5 Hz",
            "amplitude": "100 mV",
            "type": "Intermittent (0.5-1.2s, 1.8-2.3s)",
            "nature": "Unknown (possibly equipment or natural)",
            "detection": "Best seen in spectrogram"
        },
        {
            "name": "Lightning Sferics",
            "frequency": "Broadband (200-2000 Hz)",
            "amplitude": "Variable (up to 1V peak)",
            "type": "Transient impulses at 0.3s, 1.1s, 2.1s, 2.7s",
            "nature": "Natural (atmospheric)",
            "detection": "Short FFT windows, high time resolution"
        },
        {
            "name": "Mystery Chirp",
            "frequency": "150→50 Hz sweep",
            "amplitude": "150 mV",
            "type": "Single occurrence (1.4-1.7s)",
            "nature": "Unknown (possibly biological or equipment)",
            "detection": "Clear in spectrogram with fine time resolution"
        },
        {
            "name": "Background Noise",
            "frequency": "Broadband",
            "amplitude": "~50 mV RMS (varying)",
            "type": "Continuous with slow amplitude modulation",
            "nature": "Mixed (thermal + environmental)",
            "detection": "Statistical analysis of noise floor"
        }
    ]
    
    for i, comp in enumerate(components, 1):
        print(f"Component {i}: {comp['name']}")
        print(f"  • Frequency: {comp['frequency']}")
        print(f"  • Amplitude: {comp['amplitude']}")
        print(f"  • Type: {comp['type']}")
        print(f"  • Nature: {comp['nature']}")
        print(f"  • Best Detection: {comp['detection']}")
        print()
    
    print("🔧 OPTIMAL PROCESSING PARAMETERS:")
    print("For weak continuous signals (8.7 Hz):")
    print("  • FFT Size: 16384-65536 samples")
    print("  • Window: Blackman (minimal leakage)")
    print("  • Averaging: 50-100 frames")
    print()
    print("For transient detection (lightning):")
    print("  • FFT Size: 256-512 samples")
    print("  • Window: Hann")
    print("  • Overlap: 90% for smooth tracking")
    print()
    print("For general monitoring:")
    print("  • FFT Size: 2048-4096 samples")
    print("  • Window: Hann or Hamming")
    print("  • Overlap: 50-75%")
    print()
    print("💡 KEY INSIGHTS:")
    print("1. The 8.7 Hz component requires patience - it's 40 dB below the noise")
    print("2. Power line frequency drift indicates grid instability or measurement issues")
    print("3. Lightning sferics show frequency dispersion (higher frequencies arrive first)")
    print("4. The mystery chirp could be biological, mechanical, or electronic in origin")
    print("5. Multiple processing configurations are needed for complete analysis")

reveal_solution()

📋 MYSTERY SIGNAL ANALYSIS SOLUTION
🎯 IDENTIFIED COMPONENTS:
Component 1: Natural Resonance
  • Frequency: 8.7 Hz
  • Amplitude: 20 mV
  • Type: Continuous
  • Nature: Natural (possibly Schumann-like)
  • Best Detection: Requires long FFT (16384+) and averaging

Component 2: Power Line Fundamental
  • Frequency: 60 Hz (with ±0.5 Hz drift)
  • Amplitude: 800 mV
  • Type: Continuous with frequency modulation
  • Nature: Man-made interference
  • Best Detection: Easily visible in any FFT size

Component 3: Power Line Harmonics
  • Frequency: 120 Hz, 180 Hz
  • Amplitude: 300 mV, 150 mV
  • Type: Continuous
  • Nature: Man-made (harmonic distortion)
  • Best Detection: Clear in frequency domain

Component 4: Unknown Narrowband Signal
  • Frequency: 73.5 Hz
  • Amplitude: 100 mV
  • Type: Intermittent (0.5-1.2s, 1.8-2.3s)
  • Nature: Unknown (possibly equipment or natural)
  • Best Detection: Best seen in spectrogram

Component 5: Lightning Sferics
  • Frequency: Broadband (200-2000 Hz)
  • 

## Part 5: What's Next? - Your Journey Forward

### Immediate Next Steps

Now that you've mastered the fundamentals, here are the logical next areas to explore:

#### 1. **Digital Filtering** 📊
- **FIR and IIR Filter Design**: Learn to create custom filters for specific frequency bands
- **Adaptive Filtering**: Filters that adjust themselves based on signal characteristics
- **Matched Filtering**: Optimal detection of known signals in noise
- **Resources**: Start with `scipy.signal` filter design tools

#### 2. **Advanced Spectral Analysis** 🌈
- **Parametric Methods**: MUSIC, ESPRIT for super-resolution frequency estimation
- **Time-Frequency Distributions**: Wigner-Ville, wavelet transforms
- **Cepstral Analysis**: For pitch detection and echo removal
- **Higher-Order Spectra**: Bispectrum, trispectrum for nonlinear analysis

#### 3. **Array Signal Processing** 📡
- **Beamforming**: Focus on signals from specific directions
- **Direction of Arrival (DOA)**: Locate signal sources in space
- **Blind Source Separation**: Unmix multiple signals (ICA, NMF)
- **MIMO Systems**: Multiple-input multiple-output processing

#### 4. **Machine Learning for Signals** 🤖
- **Feature Extraction**: Convert signals to ML-ready features
- **Anomaly Detection**: Find unusual patterns automatically
- **Classification**: Identify signal types and sources
- **Deep Learning**: CNNs for spectrograms, RNNs for time series

#### 5. **Real-Time Implementation** ⚡
- **Streaming Processing**: Handle continuous data flows
- **GPU Acceleration**: CUDA/OpenCL for massive parallelism
- **FPGA/DSP Chips**: Hardware acceleration
- **Embedded Systems**: Run DSP on microcontrollers

### Advanced Topics by Domain

#### For ULF/VLF Research:
- Ionospheric propagation modeling
- Whistler detection and analysis  
- Schumann resonance tracking
- Lightning location systems

#### For Audio/Music:
- Pitch detection and tracking
- Audio effects (reverb, compression)
- Music information retrieval
- Speech recognition preprocessing

#### For Communications:
- Modulation/demodulation
- Channel estimation
- Error correction coding
- Software-defined radio (SDR)

#### For Biomedical:
- ECG/EEG analysis
- Medical imaging (MRI, ultrasound)
- Physiological signal monitoring
- Brain-computer interfaces

## Glossary of Essential DSP Terms

Here's your reference guide to the key concepts from this course:

| Term | Definition | Module |
|---|---|---|
| Aliasing | Distortion caused by sampling below the Nyquist rate, causing high frequencies to appear as lower frequencies | 1 |
| Coherence | Measure of correlation between two signals at each frequency (0=uncorrelated, 1=perfectly correlated) | 6 |
| DFT/FFT | Discrete/Fast Fourier Transform - algorithms to convert time-domain signals to frequency domain | 1 |
| Frequency Bin | Each frequency point in an FFT output, spaced by fs/N Hz | 1 |
| Frequency Resolution | Minimum frequency difference that can be resolved, equals fs/N Hz | 1 |
| Hop Size | Number of samples to advance between successive FFT frames in STFT | 4 |
| Leakage | Spreading of energy across frequency bins due to finite-length FFT | 2 |
| Noise Floor | Background noise level in a spectrum, sets detection threshold | 5 |
| Nyquist Frequency | Half the sampling rate, the maximum frequency that can be represented | 1 |
| Overlap | Percentage of samples shared between consecutive FFT frames | 4 |
| Parseval's Theorem | Energy in time domain equals energy in frequency domain | 3 |
| Periodicity Assumption | FFT assumes the signal repeats forever, causing edge artifacts | 2 |
| Phase | Angular component of complex frequency domain values, indicates timing | 1 |
| Power Spectral Density | Power per unit frequency (V²/Hz), independent of FFT size | 3 |
| Power Spectrum | Total power at each frequency (V²), depends on FFT size | 3 |
| Sampling Rate | Number of samples per second (Hz), determines frequency range | 1 |
| Scalloping Loss | Amplitude reduction when signal frequency falls between bins | 2 |
| SNR | Signal-to-Noise Ratio, measure of signal strength vs background noise | 5 |
| Spectral Averaging | Combining multiple FFTs to reduce noise and improve estimates | 5 |
| Spectrogram | Visual representation of STFT showing frequency content over time | 4 |
| STFT | Short-Time Fourier Transform, applies FFT to windowed segments | 4 |
| Time Resolution | Minimum time difference that can be resolved, related to window length | 1 |
| Window Function | Weighting function applied before FFT to reduce spectral leakage | 2 |
| Zero Padding | Adding zeros to increase FFT size, interpolates spectrum but doesn't improve resolution | 1 |

**📚 Total Terms:** 24  
**📖 Tip:** Keep this right next to your notebooks for quick reference.
